In [37]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv()
if os.getenv("GROQ_API_KEY"):
    print("API Key Loaded")

API Key Loaded


In [36]:
llm = ChatGroq(model = "openai/gpt-oss-120b")

### **Tools**

In [15]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.tools import tool

@tool
def search_wikipedia(query:str) ->str:
    """This Tool Is For Search In Wikipedia"""
    search = WikipediaQueryRun(api_wrapper = WikipediaAPIWrapper())
    return search.invoke(query)

### **Arxiv Query Tool**

In [17]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper
from langchain_core.tools import tool

@tool
def arxiv_tool(query:str) ->str:
    """This Fuction Is For Research Paper Purpose"""
    search = ArxivQueryRun(api_wrapper = ArxivAPIWrapper())
    return search.invoke(query)

### **DuckDuckGo Search**

In [19]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

@tool
def duckduck_tool(query:str) ->str:
    """This Fuction Is For search on duckduckgo for a latest news Purpose"""
    search = DuckDuckGoSearchRun(description = "This Is For Search On duckduckgo for a latest news")
    return search.invoke(query)

### **Tool Binding**

In [33]:
tools = [search_wikipedia,arxiv_tool,duckduck_tool]
llm_with_tools = llm.bind_tools(tools)

### **LangGraph Creation**

### Create Schema

In [24]:
from typing import TypedDict, List

class graph_schema(TypedDict):
    messages:List

### Create Node Function

In [28]:
def llm_node(state:graph_schema) ->graph_schema:
    messages = state["messages"]
    
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system","You Are A HelpFull Assistant That Can Use Tools To Answer Questions"),
            ("human","{input}")
        ]
    )
    
    chain = prompt | llm_with_tools
    response = chain.invoke({"input":messages})
    
    state["messages"] = messages + [response]
    
    return state

In [40]:
# from langgraph.prebuilt import ToolNode

def tool_node(state:graph_schema) ->graph_schema:
    messages = state["messages"]
    
    tools_by_name = {tool.name:tool for tool in tools}
    
    tool_results = []
    
    for tool in messages[-1].tool_calls:
        tool = tools_by_name[tool["name"]]
        
        observation = tool.invoke(tool["arguments"])
        
        tool_results.append(ToolMessage(content = observation, tool_call_id = tool["id"]))
    
    state["messages"] = messages + tool_results
    return state

### Create State Graph

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(graph_schema)

graph.add_node("llm_node",llm_node)
graph.add_node("tool_node",tool_node)

